# Самостоятельная работа. Автокодировщик для MNIST

Цель работы — обучить полносвязный автокодировщик восстанавливать изображения рукописных цифр.
Кодировщик должен сжимать каждое изображение ровно до двух чисел.
Постройте визуализацию этих кодов и исследуйте ответы декодировщика.

## Набор данных

MNIST содержит изображения цифр от 0 до 9 размером $28\times28$ пикселей.
Исходная обучающая часть включает 60 000 изображений, тестовая — 10 000 [1].
Таблица `df` хранит обучающую часть, таблица `df_test` — тестовую.
Столбцы `pixel1`–`pixel784` содержат яркости от 0 до 255.
Столбец `label` хранит цифру на изображении.

Модель должна восстанавливать входные пиксели.
Метки цифр используйте для разбиения данных и графиков, но не передавайте их в модель или функцию потерь.

## Импорт

In [1]:
from copy import deepcopy
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from IPython.display import display

## Данные

Прочитаем архив MNIST, используемый библиотекой Keras [2].
Для чтения архива достаточно NumPy.
Обе части запишем в отдельные таблицы без изменения исходного разбиения.

In [2]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
archive_path = data_dir / "mnist.npz"

if not archive_path.exists():
    url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
    temporary_path = data_dir / "mnist.download"
    urlretrieve(url, temporary_path)
    temporary_path.replace(archive_path)

with np.load(archive_path, allow_pickle=False) as archive:
    columns = [f"pixel{number}" for number in range(1, 785)]
    df = pd.DataFrame(archive["x_train"].reshape(-1, 784), columns=columns)
    df.insert(0, "label", archive["y_train"])
    df_test = pd.DataFrame(archive["x_test"].reshape(-1, 784), columns=columns)
    df_test.insert(0, "label", archive["y_test"])

In [3]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df.shape

(60000, 785)

In [5]:
df_test.shape

(10000, 785)

## 1. Предпросмотр данных

Покажите несколько строк `df`, размеры обеих таблиц, типы столбцов и число пропусков.
Выведите число изображений каждой цифры в `df`.
Покажите не меньше десяти обучающих изображений и подпишите их метки.
Достаточно таблиц и графиков, письменный вывод не требуется.

Подсказка. Для показа изображения выберите только пиксели одной строки.
Верните им форму `(28, 28)` через `reshape`.
Для всех изображений используйте одинаковый диапазон яркости.

## 2. Подготовка данных, Dataset и DataLoader

### 2.1. Разделение и преобразование пикселей

Выделите из `df` обучающую и валидационную части в пропорции 80/20.
Сохраните доли цифр через `stratify` и задайте `random_state`.
Таблицу `df_test` оставьте для итогового тестирования.
Не смешивайте её с `df`.

Отделите пиксели от меток.
Пиксели переведите в `float32` и разделите на 255.
Метки сохраните отдельно как целые числа.
Во всех частях должен остаться одинаковый порядок пикселей.

Подсказка. Нормирование можно выполнить здесь или внутри `Dataset`.
Делите пиксели на 255 только один раз.
Вектор одного изображения должен содержать 784 значения, а не 785.

### 2.2. Собственный Dataset

Напишите класс с методами `__init__`, `__len__` и `__getitem__`.
В конструкторе сохраните пиксели и метки как тензоры.
Пусть `__getitem__` возвращает вектор пикселей и метку одного изображения.
Создайте отдельные объекты для `train`, `valid` и `test`.

Подсказка. Все пиксели одной части имеют форму `(N, 784)` и тип `float32`.
Метки имеют форму `(N,)` и тип `long`.
Целевой ответ для восстановления уже содержится во входном изображении.
Хранить вторую копию всех пикселей не требуется.

### 2.3. Загрузчики

Создайте загрузчики для обучения, валидации и тестирования.
Перемешивание включите только для обучения.
Добавьте отдельный загрузчик обучающей части без перемешивания для итоговой оценки.
Получите один пакет и покажите формы и типы его тензоров.

Подсказка. Можно начать с пакета из 256 изображений.
В цикле обучения метку цифры можно принять в переменную `_`.
Для графиков она понадобится позже.

## 3. Функция потерь

Выберите функцию потерь для восстановления непрерывных значений пикселей.
Создайте объект `loss_function`.
В качестве целевого ответа используйте входное изображение.

Подсказка. Для первого варианта подходит `nn.MSELoss()`.
Ей передают восстановленные и исходные пиксели формы `(B, 784)`.
Метки цифр и `CrossEntropyLoss` в этой задаче не нужны.

## 4. Архитектура модели

Создайте отдельные `encoder` и `decoder` через `nn.Sequential`.
Объедините их в `autoencoder = nn.Sequential(encoder, decoder)`.
Используйте полносвязные слои и функции активации без свёрток.

Кодировщик должен преобразовывать 784 пикселя ровно в два числа.
Декодировщик должен получать только эти два числа и возвращать 784 значения.
Число скрытых слоёв и их ширину выберите сами.
Последний слой кодировщика оставьте без активации.
На выходе декодировщика используйте `Sigmoid`.

Выберите устройство и перенесите на него модель.
Покажите архитектуру, число параметров и формы входа, кода и восстановления для одного пакета.

Подсказка. Ожидаемые формы равны `(B, 784)`, `(B, 2)` и `(B, 784)`.
Два измерения должны быть частью обучаемой сети, а не результатом PCA или t-SNE после обучения.
Не отделяйте код от графа вычислений через `detach()` внутри обучаемой цепочки.

## 5. Оптимизатор

Выберите оптимизатор и задайте шаг обучения.
Передайте ему параметры всего автокодировщика.

Подсказка. Можно начать с Adam и шага 0.001.
Сначала перенесите модель на выбранное устройство, затем создайте оптимизатор.
Оптимизатор должен обновлять и кодировщик, и декодировщик.

## 6. Цикл обучения и валидации

Напишите цикл по эпохам и вложенный цикл по обучающим пакетам.
Внутри очистите градиенты, получите восстановление, вычислите потерю и обновите веса.
После каждой эпохи выполните отдельный проход по `valid` без обновления параметров.

Сохраняйте MSE и MAE для `train` и `valid` по эпохам.
При уменьшении валидационного MSE копируйте состояние модели и номер эпохи.
После обучения восстановите лучшие параметры.
Архитектуру и настройки выбирайте по `valid`, не используя `test`.

Подсказка. Для обучения вызывайте `train()`.
При валидации используйте `eval()` и `torch.no_grad()`.
Вычисляйте ошибку между восстановлением и входом, а не между восстановлением и меткой.
Средние ошибки пакетов взвешивайте по их размерам.
Состояние модели можно копировать через `deepcopy(autoencoder.state_dict())`.

## 7. Цикл тестирования

Напишите отдельный цикл по тестовому загрузчику.
Получите двумерные коды и восстановления с выбранными параметрами.
Соберите исходные изображения, восстановления, коды и метки для всей тестовой части.
Вычислите средний тестовый MSE.

Подсказка. Используйте `eval()` и `torch.no_grad()`.
Собирайте все значения в одном цикле, чтобы не потерять соответствие объектов.
Сначала можно добавлять пакеты в списки, затем объединить их через `torch.cat`.
Перед переводом в NumPy перенесите тензоры на CPU.

## 8. Метрики и история обучения

### 8.1. Итоговые метрики

Получите ответы выбранной модели на `train` без обновления весов.
Рассчитайте MSE, RMSE и MAE отдельно для обучающей и тестовой частей.
Соберите результаты в таблицу.
Укажите в названиях или подписях, что пиксели приведены к диапазону $[0,1]$.

Для сравнения вычислите среднее изображение только по `train`.
Используйте его как одинаковый ответ для всех объектов.
Добавьте MSE такого ответа на `train` и `test` в итоговую таблицу.

Подсказка. MSE и MAE усредняются по всем изображениям и всем 784 пикселям.
RMSE вычисляйте как корень из общего MSE, а не как среднее RMSE отдельных пакетов.
Метрики выбранной модели пересчитайте заново, не берите их из последней строки истории.

### 8.2. Графики по эпохам

Постройте график MSE на `train` и `valid`.
Отдельно покажите MAE на этих частях.
Отметьте выбранную эпоху и подпишите оси.
Тестовую ошибку по эпохам не вычисляйте.

Подсказка. Значения `train`, накопленные при обновлении весов, относятся к разным состояниям модели внутри эпохи.
Для оценки `train` при одном состоянии весов можно сделать дополнительный проход после эпохи.

## 9. Визуализация результатов

### 9.1. Исходные и восстановленные изображения

Покажите не меньше десяти пар тестовых изображений.
В каждой паре разместите рядом исходное изображение и его восстановление.
Подпишите цифры из разметки.
Сохраните одинаковую шкалу яркости от 0 до 1.

Подсказка. Выберите первые изображения или случайные индексы с фиксированным seed.
Не отбирайте только примеры с минимальной ошибкой.
Восстановленные векторы верните к форме `(28, 28)`.

### 9.2. Ошибка каждого изображения

Вычислите MSE отдельно для каждого тестового изображения.
Постройте гистограмму этих ошибок.
Покажите несколько пар с наибольшей ошибкой и подпишите их MSE.

Подсказка. При форме массива `(N, 784)` усреднение по `axis=1` оставит одну ошибку на изображение.
Для выбора трудных примеров можно использовать `argsort`.
Подпишите, что эти изображения специально отобраны по большой ошибке.

### 9.3. Двумерные коды

Постройте диаграмму рассеяния кодов тестовых изображений.
По горизонтали отложите первый выход `encoder`, по вертикали — второй.
Обозначьте десять цифр разными цветами или маркерами и добавьте легенду.

Подсказка. Используйте массив формы `(N, 2)`, полученный при тестировании.
Дополнительные методы снижения размерности не применяйте.
Для читаемости можно показать случайную часть точек.
Разделение цифр на непересекающиеся группы не является обязательным результатом обучения восстановлению.

### 9.4. Ошибки на плоскости кодов

Постройте ещё одну диаграмму тех же кодов.
Теперь обозначьте цветом MSE соответствующего изображения.
Добавьте цветовую шкалу с подписью метрики.

Подсказка. Для каждой точки должны совпадать индекс кода и индекс ошибки.
Этот график показывает ошибки наблюдаемых изображений, а не ошибки произвольных точек плоскости.

### 9.5. Карта ответов декодировщика

Получите коды обучающих изображений с выбранными параметрами.
По ним задайте диапазоны двух координат.
Постройте регулярную сетку не меньше $10\times10$ точек.
Передайте точки в `decoder` и расположите полученные изображения в той же сетке.
Подпишите направления изменения двух координат.

Подсказка. Границы можно взять по 2-му и 98-му процентилям обучающих кодов.
Используйте `meshgrid` и объедините координаты в массив формы `(K, 2)`.
Верхняя строка карты должна соответствовать большим значениям второй координаты.
Пустые области между кодами реальных изображений тоже попадут в сетку, но правдоподобные цифры в них не гарантируются.

### 9.6. Линии уровня ошибки для одного изображения

Выберите одно тестовое изображение $x^{(0)}$.
На более плотной сетке кодов вычислите ошибку между ответом декодировщика и этим изображением.
Постройте линии уровня полученной функции и отметьте код выбранного изображения.
Рядом отдельным рисунком покажите само изображение.

$$E(z_1,z_2)=\frac{1}{784}\sum_{p=1}^{784}\left(g_\varphi(z_1,z_2)_p-x^{(0)}_p\right)^2.$$

Подсказка. Для каждой точки сетки нужен один MSE по 784 пикселям.
Массив ошибок верните к форме сетки и передайте в `contour`.
При нехватке памяти обрабатывайте точки пакетами.
Эти линии относятся только к выбранному изображению, а не к вероятностям цифр или ошибке всей выборки.

### 9.7. Переход между двумя изображениями

Выберите два тестовых изображения с разными цифрами.
Постройте не меньше девяти точек на отрезке между их кодами.
Декодируйте точки и покажите последовательность изображений.
Подпишите значение параметра $t$ под каждым изображением.

$$z(t)=(1-t)z_a+tz_b,\qquad 0\leq t\leq1.$$

Подсказка. Значения $t$ удобно получить через `linspace`.
Меняйте координаты кода, а не смешивайте пиксели двух исходных изображений.
На концах последовательности будут восстановления, а не исходные изображения.

## Источники

[1] [MNIST. Описание состава и формата данных](https://keras.io/api/datasets/mnist/).

[2] [Keras. Адрес исходного архива MNIST](https://github.com/keras-team/keras/blob/master/keras/src/datasets/mnist.py).